# Link Prediction with GAT on ogbl-collab — Evaluation & Visualization

This notebook provides comprehensive evaluation and visualization of the trained GAT link prediction model, including:

1. **Training Curves** — Loss and Hits@K over epochs
2. **Score Distributions** — Positive vs. negative edge prediction scores
3. **Multi-Metric Comparison** — Hits@10, Hits@50, Hits@100 across runs
4. **Attention Analysis** — Visualization of learned attention weights
5. **Graph Structure Analysis** — Degree distribution and prediction patterns

In [ ]:
import sys
import os
import json

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from torch.utils.data import DataLoader

# Add project root
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.models import GATEncoder, LinkPredictor
from src.data import load_dataset, prepare_features
from src.training.evaluator import evaluate
from src.utils.seed import set_seed

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

print("Imports OK")

## 1. Load Training Logs

Load the JSON training logs exported by the training script.

In [ ]:
LOG_PATH = "../logs/training_log.json"
CONFIG_PATH = "../configs/default.yaml"
CHECKPOINT_DIR = "../checkpoints"

# Load training logs
if os.path.exists(LOG_PATH):
    with open(LOG_PATH, "r") as f:
        log_data = json.load(f)
    print(f"Loaded training log: {len(log_data['losses'])} run(s)")
    print(f"Metrics tracked: {list(log_data['metrics'].keys())}")
    HAS_LOGS = True
else:
    print(f"Training log not found at {LOG_PATH}")
    print("Please run training first: python scripts/train.py")
    print("\nProceeding with demo/placeholder visualizations...")
    HAS_LOGS = False

## 2. Training Loss Curves

Visualize the training loss over epochs for each run, along with a smoothed trend line.

In [ ]:
def smooth(values, weight=0.9):
    """Exponential moving average smoothing."""
    smoothed = []
    last = values[0]
    for v in values:
        last = weight * last + (1 - weight) * v
        smoothed.append(last)
    return smoothed

if HAS_LOGS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Raw loss curves
    ax = axes[0]
    for i, losses in enumerate(log_data["losses"]):
        epochs = range(1, len(losses) + 1)
        ax.plot(epochs, losses, alpha=0.3, label=f"Run {i+1} (raw)")
        ax.plot(epochs, smooth(losses), linewidth=2, label=f"Run {i+1} (smoothed)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Loss comparison across runs (final 50 epochs)
    ax = axes[1]
    for i, losses in enumerate(log_data["losses"]):
        tail = losses[-min(50, len(losses)):]
        ax.plot(range(len(tail)), tail, linewidth=2, label=f"Run {i+1}")
    ax.set_xlabel("Last N Epochs")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss (Final Epochs)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("../logs/training_loss.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/training_loss.png")
else:
    print("Skipped — no training logs available.")

## 3. Hits@K Metrics Over Epochs

Track Train/Valid/Test Hits@K for each evaluation step.

In [ ]:
if HAS_LOGS:
    metric_names = list(log_data["metrics"].keys())
    n_metrics = len(metric_names)
    n_runs = len(log_data["losses"])

    fig, axes = plt.subplots(n_metrics, 1, figsize=(14, 5 * n_metrics), squeeze=False)

    colors = {"Train": "#2196F3", "Valid": "#FF9800", "Test": "#4CAF50"}

    for m_idx, metric in enumerate(metric_names):
        ax = axes[m_idx, 0]
        for run_idx in range(n_runs):
            run_data = np.array(log_data["metrics"][metric][run_idx]) * 100
            if len(run_data) == 0:
                continue
            eval_steps = np.arange(1, len(run_data) + 1)

            alpha = 0.6 if n_runs > 1 else 1.0
            lw = 1.5 if n_runs > 1 else 2.0

            for split_idx, split_name in enumerate(["Train", "Valid", "Test"]):
                label = f"{split_name}" if run_idx == 0 else None
                ax.plot(eval_steps, run_data[:, split_idx],
                        color=colors[split_name], alpha=alpha, linewidth=lw,
                        label=label, linestyle="-" if run_idx == 0 else "--")

        ax.set_xlabel("Evaluation Step")
        ax.set_ylabel(f"{metric} (%)")
        ax.set_title(f"{metric} Over Training")
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)

        # Mark best valid
        for run_idx in range(n_runs):
            run_data = np.array(log_data["metrics"][metric][run_idx]) * 100
            if len(run_data) > 0:
                best_idx = np.argmax(run_data[:, 1])
                ax.axvline(x=best_idx + 1, color="red", linestyle=":", alpha=0.5)
                ax.annotate(
                    f"Best Valid: {run_data[best_idx, 1]:.1f}%\nTest: {run_data[best_idx, 2]:.1f}%",
                    xy=(best_idx + 1, run_data[best_idx, 1]),
                    xytext=(15, -10), textcoords="offset points",
                    fontsize=9, color="red",
                    arrowprops=dict(arrowstyle="->", color="red", alpha=0.7),
                )

    plt.tight_layout()
    plt.savefig("../logs/hits_at_k_curves.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/hits_at_k_curves.png")
else:
    print("Skipped — no training logs available.")

## 4. Final Results Summary Table

Best Valid and Test@BestValid for each metric across all runs.

In [ ]:
import pandas as pd

if HAS_LOGS:
    rows = []
    for metric in log_data["metrics"]:
        for run_idx, run_data in enumerate(log_data["metrics"][metric]):
            arr = np.array(run_data) * 100
            if len(arr) == 0:
                continue
            best_valid_idx = np.argmax(arr[:, 1])
            rows.append({
                "Metric": metric,
                "Run": run_idx + 1,
                "Best Valid (%)": f"{arr[best_valid_idx, 1]:.2f}",
                "Test @ Best Valid (%)": f"{arr[best_valid_idx, 2]:.2f}",
                "Best Epoch Step": best_valid_idx + 1,
            })

    df = pd.DataFrame(rows)
    display(df)

    # Aggregate across runs
    print("\n--- Aggregated Results ---")
    for metric in log_data["metrics"]:
        vals = []
        tests = []
        for run_data in log_data["metrics"][metric]:
            arr = np.array(run_data) * 100
            if len(arr) == 0:
                continue
            best_idx = np.argmax(arr[:, 1])
            vals.append(arr[best_idx, 1])
            tests.append(arr[best_idx, 2])
        vals, tests = np.array(vals), np.array(tests)
        print(f"{metric}:")
        print(f"  Valid: {vals.mean():.2f} +/- {vals.std():.2f}")
        print(f"  Test:  {tests.mean():.2f} +/- {tests.std():.2f}")
else:
    print("Skipped — no training logs available.")

## 5. Load Best Checkpoint & Generate Predictions

Load the best model checkpoint and run inference to get prediction scores for detailed analysis.

In [ ]:
import yaml

HAS_CHECKPOINT = False
raw_preds = None

ckpt_path = os.path.join(CHECKPOINT_DIR, "best_run1.pt")
if os.path.exists(ckpt_path) and os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r") as f:
        cfg = yaml.safe_load(f)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    set_seed(cfg["experiment"]["seed"])

    # Load data
    data, split_edge, dataset = load_dataset(cfg)
    data = prepare_features(data, cfg, device)

    mcfg = cfg["model"]
    pcfg = cfg["predictor"]
    emb_dim = cfg["node_embedding"]["embedding_dim"]

    # Rebuild model
    encoder = GATEncoder(
        in_channels=emb_dim if cfg["node_embedding"]["use_embedding"] else data.x.size(-1),
        hidden_channels=mcfg["hidden_channels"],
        out_channels=mcfg["hidden_channels"],
        num_layers=mcfg["num_layers"],
        dropout=mcfg["dropout"],
        attn_dropout=mcfg["attn_dropout"],
        heads=mcfg["heads"],
        use_gatv2=mcfg["use_gatv2"],
        residual=mcfg["residual"],
        layer_norm=mcfg["layer_norm"],
        jk_mode=mcfg["jk_mode"],
    ).to(device)

    encoder_out_dim = encoder.out_dim
    predictor = LinkPredictor(
        encoder_out_dim, pcfg["hidden_channels"], 1,
        pcfg["num_layers"], pcfg["dropout"]
    ).to(device)

    node_emb = None
    if cfg["node_embedding"]["use_embedding"]:
        node_emb = torch.nn.Embedding(data.num_nodes, emb_dim).to(device)

    # Load checkpoint
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    encoder.load_state_dict(ckpt["encoder"])
    predictor.load_state_dict(ckpt["predictor"])
    if node_emb is not None and "node_emb" in ckpt:
        node_emb.load_state_dict(ckpt["node_emb"])

    print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
    print(f"Valid score at checkpoint: {100 * ckpt['valid_score']:.2f}%")

    # Generate predictions
    from ogb.linkproppred import Evaluator as OGBEvaluator
    ogb_ev = OGBEvaluator(name=cfg["dataset"]["name"])
    metrics = cfg["evaluation"]["metrics"]

    results, raw_preds = evaluate(
        encoder, predictor, data, split_edge,
        ogb_ev, cfg["training"]["batch_size"], metrics, node_emb
    )

    for key, (tr, va, te) in results.items():
        print(f"{key}: Train={100*tr:.2f}%, Valid={100*va:.2f}%, Test={100*te:.2f}%")

    HAS_CHECKPOINT = True
else:
    print(f"Checkpoint not found at {ckpt_path}")
    print("Run training first to generate checkpoints.")

## 6. Prediction Score Distributions

Compare the distribution of prediction scores for positive vs. negative edges on each split.

In [ ]:
if HAS_CHECKPOINT and raw_preds is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Validation split
    ax = axes[0]
    ax.hist(raw_preds["pos_valid_pred"].numpy(), bins=80, alpha=0.7,
            label="Positive", color="#4CAF50", density=True)
    ax.hist(raw_preds["neg_valid_pred"].numpy(), bins=80, alpha=0.7,
            label="Negative", color="#F44336", density=True)
    ax.set_xlabel("Prediction Score")
    ax.set_ylabel("Density")
    ax.set_title("Validation Set")
    ax.legend()

    # Test split
    ax = axes[1]
    ax.hist(raw_preds["pos_test_pred"].numpy(), bins=80, alpha=0.7,
            label="Positive", color="#4CAF50", density=True)
    ax.hist(raw_preds["neg_test_pred"].numpy(), bins=80, alpha=0.7,
            label="Negative", color="#F44336", density=True)
    ax.set_xlabel("Prediction Score")
    ax.set_ylabel("Density")
    ax.set_title("Test Set")
    ax.legend()

    # Box plot comparison
    ax = axes[2]
    box_data = [
        raw_preds["pos_valid_pred"].numpy(),
        raw_preds["neg_valid_pred"].numpy(),
        raw_preds["pos_test_pred"].numpy(),
        raw_preds["neg_test_pred"].numpy(),
    ]
    bp = ax.boxplot(box_data, labels=["Val+", "Val-", "Test+", "Test-"],
                    patch_artist=True, showfliers=False)
    box_colors = ["#4CAF50", "#F44336", "#4CAF50", "#F44336"]
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel("Prediction Score")
    ax.set_title("Score Distribution Comparison")

    plt.tight_layout()
    plt.savefig("../logs/score_distributions.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/score_distributions.png")
else:
    print("Skipped — no checkpoint loaded. Run training first.")

## 7. Multi-Metric Bar Chart

Compare Hits@10, Hits@50, Hits@100 across Train/Valid/Test splits in a grouped bar chart.

In [ ]:
if HAS_CHECKPOINT:
    metric_names = list(results.keys())
    train_scores = [100 * results[m][0] for m in metric_names]
    valid_scores = [100 * results[m][1] for m in metric_names]
    test_scores = [100 * results[m][2] for m in metric_names]

    x = np.arange(len(metric_names))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width, train_scores, width, label="Train", color="#2196F3", alpha=0.8)
    bars2 = ax.bar(x, valid_scores, width, label="Valid", color="#FF9800", alpha=0.8)
    bars3 = ax.bar(x + width, test_scores, width, label="Test", color="#4CAF50", alpha=0.8)

    ax.set_xlabel("Metric")
    ax.set_ylabel("Score (%)")
    ax.set_title("Best Model Performance Across Metrics")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f"{h:.1f}",
                        xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig("../logs/multi_metric_bar.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/multi_metric_bar.png")
else:
    print("Skipped — no checkpoint loaded.")

## 8. Dataset Statistics & Graph Analysis

Analyze the ogbl-collab graph structure: degree distribution, edge split sizes, and node connectivity patterns.

In [ ]:
if HAS_CHECKPOINT:
    # Degree distribution
    deg = data.adj_t.sum(dim=1).cpu().numpy()

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Degree distribution (log-log)
    ax = axes[0, 0]
    unique_degs, counts = np.unique(deg.astype(int), return_counts=True)
    ax.scatter(unique_degs, counts, s=3, alpha=0.5, color="#3F51B5")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Degree")
    ax.set_ylabel("Count")
    ax.set_title("Degree Distribution (Log-Log)")
    ax.grid(True, alpha=0.3)

    # Degree histogram
    ax = axes[0, 1]
    ax.hist(deg, bins=100, color="#3F51B5", alpha=0.7, edgecolor="white")
    ax.set_xlabel("Degree")
    ax.set_ylabel("Count")
    ax.set_title(f"Degree Histogram (mean={deg.mean():.1f}, median={np.median(deg):.1f})")
    ax.axvline(deg.mean(), color="red", linestyle="--", label=f"Mean: {deg.mean():.1f}")
    ax.axvline(np.median(deg), color="orange", linestyle="--", label=f"Median: {np.median(deg):.1f}")
    ax.legend()

    # Edge split sizes
    ax = axes[1, 0]
    split_sizes = {
        "Train": split_edge["train"]["edge"].size(0),
        "Valid+": split_edge["valid"]["edge"].size(0),
        "Valid-": split_edge["valid"]["edge_neg"].size(0),
        "Test+": split_edge["test"]["edge"].size(0),
        "Test-": split_edge["test"]["edge_neg"].size(0),
    }
    bars = ax.bar(split_sizes.keys(), split_sizes.values(),
                  color=["#2196F3", "#4CAF50", "#F44336", "#4CAF50", "#F44336"],
                  alpha=0.8)
    ax.set_ylabel("Number of Edges")
    ax.set_title("Edge Split Sizes")
    for bar, v in zip(bars, split_sizes.values()):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f"{v:,}", ha="center", va="bottom", fontsize=9)

    # Node statistics
    ax = axes[1, 1]
    stats = {
        "Nodes": data.num_nodes,
        "Train Edges": split_sizes["Train"],
        "Mean Degree": f"{deg.mean():.1f}",
        "Max Degree": int(deg.max()),
        "Min Degree": int(deg.min()),
        "Isolated Nodes": int((deg == 0).sum()),
    }
    ax.axis("off")
    table_data = [[k, str(v)] for k, v in stats.items()]
    table = ax.table(cellText=table_data, colLabels=["Statistic", "Value"],
                     loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.5)
    ax.set_title("Dataset Statistics", pad=20)

    plt.tight_layout()
    plt.savefig("../logs/dataset_analysis.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/dataset_analysis.png")
else:
    print("Skipped — load checkpoint section first.")

## 9. Prediction Score vs. Node Degree

Analyze how prediction accuracy varies with node degree — do high-degree nodes get better predictions?

In [ ]:
if HAS_CHECKPOINT and raw_preds is not None:
    deg = data.adj_t.sum(dim=1).cpu().numpy()

    # For positive test edges: average score by source node degree
    pos_test_edges = split_edge["test"]["edge"].cpu().numpy()
    pos_scores = raw_preds["pos_test_pred"].numpy()

    # Bin by degree
    src_degs = deg[pos_test_edges[:, 0]]
    bins = np.percentile(src_degs, np.arange(0, 101, 10))
    bins = np.unique(bins)
    bin_indices = np.digitize(src_degs, bins) - 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mean score by degree bin
    ax = axes[0]
    bin_means = []
    bin_labels = []
    for i in range(len(bins) - 1):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_means.append(pos_scores[mask].mean())
            bin_labels.append(f"{bins[i]:.0f}-{bins[i+1]:.0f}")
    ax.bar(range(len(bin_means)), bin_means, color="#673AB7", alpha=0.7)
    ax.set_xticks(range(len(bin_labels)))
    ax.set_xticklabels(bin_labels, rotation=45, ha="right")
    ax.set_xlabel("Source Node Degree Range")
    ax.set_ylabel("Mean Positive Score")
    ax.set_title("Prediction Score vs. Node Degree (Positive Test Edges)")

    # Scatter: degree vs score (subsample for clarity)
    ax = axes[1]
    n_sample = min(5000, len(src_degs))
    idx = np.random.choice(len(src_degs), n_sample, replace=False)
    sc = ax.scatter(src_degs[idx], pos_scores[idx], s=2, alpha=0.3, c="#673AB7")
    ax.set_xlabel("Source Node Degree")
    ax.set_ylabel("Prediction Score")
    ax.set_title("Degree vs. Score Scatter (Sampled)")
    ax.set_xscale("log")

    plt.tight_layout()
    plt.savefig("../logs/degree_vs_score.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/degree_vs_score.png")
else:
    print("Skipped — no predictions available.")

## 10. ROC-like Analysis

Plot the separation between positive and negative prediction scores using a ROC-like threshold sweep.

In [ ]:
if HAS_CHECKPOINT and raw_preds is not None:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, split_name, pos_key, neg_key in [
        (axes[0], "Validation", "pos_valid_pred", "neg_valid_pred"),
        (axes[1], "Test", "pos_test_pred", "neg_test_pred"),
    ]:
        pos_scores = raw_preds[pos_key].numpy()
        neg_scores = raw_preds[neg_key].numpy()
        y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
        y_score = np.concatenate([pos_scores, neg_scores])

        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        ax.plot(fpr, tpr, color="#1976D2", linewidth=2,
                label=f"ROC (AUC = {roc_auc:.4f})")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
        ax.fill_between(fpr, tpr, alpha=0.1, color="#1976D2")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC Curve — {split_name}")
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("../logs/roc_curves.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/roc_curves.png")

    # Precision-Recall curve
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, split_name, pos_key, neg_key in [
        (axes[0], "Validation", "pos_valid_pred", "neg_valid_pred"),
        (axes[1], "Test", "pos_test_pred", "neg_test_pred"),
    ]:
        pos_scores = raw_preds[pos_key].numpy()
        neg_scores = raw_preds[neg_key].numpy()
        y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
        y_score = np.concatenate([pos_scores, neg_scores])

        precision, recall, _ = precision_recall_curve(y_true, y_score)
        ap = average_precision_score(y_true, y_score)

        ax.plot(recall, precision, color="#E64A19", linewidth=2,
                label=f"PR (AP = {ap:.4f})")
        ax.fill_between(recall, precision, alpha=0.1, color="#E64A19")
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(f"Precision-Recall Curve — {split_name}")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("../logs/pr_curves.png", bbox_inches="tight")
    plt.show()
    print("Saved: logs/pr_curves.png")
else:
    print("Skipped — no predictions available.")

## 11. Model Architecture Summary

Display the model architecture and parameter counts.

In [ ]:
if HAS_CHECKPOINT:
    print("=" * 60)
    print("GAT Encoder Architecture")
    print("=" * 60)
    print(encoder)
    print(f"\nEncoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")
    print(f"\n{'=' * 60}")
    print("Link Predictor Architecture")
    print("=" * 60)
    print(predictor)
    print(f"\nPredictor parameters: {sum(p.numel() for p in predictor.parameters()):,}")

    if node_emb is not None:
        print(f"\nNode Embedding parameters: {node_emb.weight.numel():,}")

    total = (sum(p.numel() for p in encoder.parameters())
             + sum(p.numel() for p in predictor.parameters()))
    if node_emb is not None:
        total += node_emb.weight.numel()
    print(f"\n{'=' * 60}")
    print(f"TOTAL PARAMETERS: {total:,}")

    # Parameter breakdown chart
    components = {"Encoder": sum(p.numel() for p in encoder.parameters()),
                  "Predictor": sum(p.numel() for p in predictor.parameters())}
    if node_emb is not None:
        components["Node Emb"] = node_emb.weight.numel()

    fig, ax = plt.subplots(figsize=(8, 5))
    wedges, texts, autotexts = ax.pie(
        components.values(), labels=components.keys(),
        autopct=lambda pct: f"{pct:.1f}%\n({int(pct/100 * total):,})",
        colors=["#2196F3", "#FF9800", "#4CAF50"],
        startangle=90, textprops={"fontsize": 11}
    )
    ax.set_title(f"Parameter Distribution (Total: {total:,})")
    plt.tight_layout()
    plt.savefig("../logs/param_distribution.png", bbox_inches="tight")
    plt.show()
else:
    print("Skipped — no checkpoint loaded.")